# Clase 194 — Versionado de datos con DVC

> Parte 4 · MLOps · Fuente: Huyen cap. 6 + docs DVC 3.x.

**Objetivo**: trackear un dataset con DVC, configurar un remote local, declarar un pipeline reproducible en `dvc.yaml` y correr experimentos sin contaminar `git log`.

> ⚠️ Este notebook ejecuta comandos shell (`!dvc ...`, `!git ...`) en un directorio temporal `/tmp/dvc_demo` (Linux/Mac) o `%TEMP%/dvc_demo` (Windows). Requiere `dvc` instalado: `pip install dvc`.

## Setup

In [ ]:
import os, shutil, subprocess, tempfile, json
from pathlib import Path
import pandas as pd
import seaborn as sns

WORK = Path(tempfile.gettempdir()) / 'dvc_demo'
REMOTE = Path(tempfile.gettempdir()) / 'dvc_remote_demo'
for p in (WORK, REMOTE):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

In [ ]:
def sh(cmd):
    """Ejecuta comando shell y muestra stdout/stderr. Devuelve returncode."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print('STDERR:', r.stderr)
    return r.returncode

sh('git --version && dvc --version')

## 1. `git init` + `dvc init`

DVC vive *encima* de git: necesita un repo git para guardar los `.dvc` pointers.

In [ ]:
sh('git init -q && git config user.email demo@local && git config user.name demo')
sh('dvc init -q')
sh('git add .dvc .dvcignore && git commit -q -m "init dvc"')
sh('ls -la .dvc/')

## 2. `dvc add` — trackear un dataset

Generamos un CSV pequeño (titanic ≈ 60 KB) y lo trackeamos. El blob va al cache local; en git solo queda un `.dvc` de ~200 B.

In [ ]:
Path('data/raw').mkdir(parents=True, exist_ok=True)
tit = sns.load_dataset('titanic')
tit.to_csv('data/raw/titanic.csv', index=False)
print(f'CSV size: {Path("data/raw/titanic.csv").stat().st_size:,} bytes')

sh('dvc add data/raw/titanic.csv')
sh('cat data/raw/titanic.csv.dvc')   # el pointer
sh('cat data/raw/.gitignore')          # DVC agrega el blob al .gitignore

In [ ]:
# Commit del puntero (NO del blob — el blob ya está en .gitignore)
sh('git add data/raw/titanic.csv.dvc data/raw/.gitignore && git commit -q -m "add titanic dataset"')
sh('git ls-files data/')   # solo .dvc y .gitignore — el CSV NO está en git

## 3. Remote y `dvc push`

Configuramos un remote local (simula S3). En producción sería `dvc remote add -d origin s3://bucket/path`.

In [ ]:
sh(f'dvc remote add -d local {REMOTE}')
sh('git add .dvc/config && git commit -q -m "add remote"')
sh('dvc push')

# El blob aparece en el remote, organizado por hash MD5 (prefijo de 2 chars)
for p in sorted(REMOTE.rglob('*'))[:10]:
    if p.is_file(): print(p.relative_to(REMOTE))

## 4. Pipeline declarativo (`dvc.yaml`)

Tres stages: `prepare → train → evaluate`. Cada uno declara `deps`, `outs` y opcionalmente `params`/`metrics`. `dvc repro` re-ejecuta solo lo que cambió.

In [ ]:
Path('src').mkdir(exist_ok=True)

Path('src/prepare.py').write_text('''\
import pandas as pd, sys
df = pd.read_csv(sys.argv[1]).dropna(subset=["age", "fare", "survived"])
df = df[["age", "fare", "pclass", "sex", "survived"]]
df["sex"] = (df["sex"] == "male").astype(int)
df.to_csv(sys.argv[2], index=False)
''')

Path('src/train.py').write_text('''\
import pandas as pd, yaml, joblib, sys
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
params = yaml.safe_load(open("params.yaml"))["train"]
df = pd.read_csv(sys.argv[1])
X, y = df.drop(columns=["survived"]), df["survived"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=params["test_size"], random_state=42)
m = LogisticRegression(C=params["C"], max_iter=1000).fit(Xtr, ytr)
joblib.dump({"model": m, "Xte": Xte, "yte": yte}, sys.argv[2])
''')

Path('src/evaluate.py').write_text('''\
import joblib, json, sys
from sklearn.metrics import accuracy_score, f1_score
b = joblib.load(sys.argv[1])
pred = b["model"].predict(b["Xte"])
json.dump({"accuracy": accuracy_score(b["yte"], pred), "f1": f1_score(b["yte"], pred)},
          open(sys.argv[2], "w"), indent=2)
''')

Path('params.yaml').write_text('train:\n  test_size: 0.2\n  C: 1.0\n')

Path('dvc.yaml').write_text('''\
stages:
  prepare:
    cmd: python src/prepare.py data/raw/titanic.csv data/processed/clean.csv
    deps:
      - data/raw/titanic.csv
      - src/prepare.py
    outs:
      - data/processed/clean.csv
  train:
    cmd: python src/train.py data/processed/clean.csv model.pkl
    deps:
      - data/processed/clean.csv
      - src/train.py
    params:
      - train.test_size
      - train.C
    outs:
      - model.pkl
  evaluate:
    cmd: python src/evaluate.py model.pkl metrics.json
    deps:
      - model.pkl
      - src/evaluate.py
    metrics:
      - metrics.json:
          cache: false
''')

print('Pipeline declarado.')

In [ ]:
sh('dvc repro')
print('--- metrics.json ---')
print(Path('metrics.json').read_text())
print('--- dvc.lock (primeras líneas) ---')
print('\n'.join(Path('dvc.lock').read_text().splitlines()[:20]))

## 5. Re-ejecución incremental

Cambiamos `C` en `params.yaml`. DVC detecta que cambió un `param` de `train` y re-ejecuta solo `train + evaluate` (no `prepare`).

In [ ]:
import yaml
p = yaml.safe_load(open('params.yaml'))
p['train']['C'] = 0.01
yaml.safe_dump(p, open('params.yaml', 'w'))

sh('dvc repro')   # observá: "Stage 'prepare' didn't change, skipping"
print(Path('metrics.json').read_text())

## 6. Experimentos con `dvc exp run`

Tres corridas variando `C`, sin tocar `git log`.

In [ ]:
sh('git add . && git commit -q -m "pipeline ready"')
for c in [0.01, 1.0, 100.0]:
    sh(f'dvc exp run -S train.C={c} --quiet')
sh('dvc exp show --no-pager --drop ".*" --keep "Experiment|C|accuracy|f1"')

## Ejercicio guiado

1. Agregá un cuarto stage `register` que copie `model.pkl` a `models/<git-sha-corto>.pkl`. Pista: usá `$(git rev-parse --short HEAD)` en el `cmd`.
2. Borrá el cache local (`rm -rf .dvc/cache data/processed model.pkl`) y reconstruí TODO con `dvc pull && dvc repro`. Confirmá que el hash final de `metrics.json` coincide con el de `dvc.lock`.
3. Agregá una imagen `confusion_matrix.png` como `outs` del stage `evaluate` y visualizala con `dvc plots show`.

## Conclusiones

- DVC desacopla **versionado lógico** (git, liviano) de **storage físico** (remote, pesado).
- `dvc.yaml` + `dvc.lock` son el contrato de reproducibilidad: mismo commit → misma corrida.
- `dvc exp` reemplaza el patrón "branch por experimento" que ensucia git.
- En producción real: el remote es S3/GCS, el CI hace `dvc pull` antes de entrenar, y `dvc.lock` se commitea junto al PR.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas de los 5 ejercicios del README. DVC es una **CLI que envuelve git**, así que los comandos reales (`dvc add`, `dvc push`, `dvc repro`, `dvc exp`) se muestran como se ejecutarían en una terminal con `dvc` instalado. Para que la celda **corra igual sin `dvc`**, cada solución ilustra el *concepto ejecutable* subyacente con la librería estándar (`hashlib` para el hash de contenido, `pyyaml` para el `dvc.yaml`, `scikit-learn` para el pipeline real). Así entendés *qué hace* DVC por dentro, no solo *cómo se llama* el comando.

### Ejercicio 1 — Setup mínimo y anatomía del `.dvc`

El comando real es `git init && dvc init && dvc add data/raw/titanic.csv`. El archivo `.dvc` resultante es solo un YAML con `md5`, `size` y `path`: DVC **hashea el contenido** del archivo y guarda ese puntero en git; el blob va al cache. Reproducimos el cálculo del `md5` exactamente como lo hace DVC (MD5 del contenido en binario) para ver que el `.dvc` no tiene nada mágico.

In [ ]:
import hashlib, tempfile, os, io, csv
from pathlib import Path

# --- comandos reales (referencia, requieren `dvc` instalado) ---
CLI = """
git init -q
dvc init -q
dvc add data/raw/titanic.csv     # crea data/raw/titanic.csv.dvc + lo mete en .gitignore
git add data/raw/titanic.csv.dvc data/raw/.gitignore
git commit -m "track raw titanic with dvc"
"""

# --- concepto ejecutable: generamos un CSV y calculamos su .dvc a mano ---
work = Path(tempfile.mkdtemp())
csv_path = work / 'titanic.csv'
rows = [['survived', 'pclass', 'age', 'fare']]
rng_vals = [(0, 3, 22.0, 7.25), (1, 1, 38.0, 71.28), (1, 3, 26.0, 7.92), (1, 1, 35.0, 53.1)]
rows += [[str(v) for v in r] for r in rng_vals]
with open(csv_path, 'w', newline='') as f:
    csv.writer(f).writerows(rows)

blob = csv_path.read_bytes()
md5 = hashlib.md5(blob).hexdigest()
dvc_pointer = {'outs': [{'md5': md5, 'size': len(blob), 'path': 'titanic.csv'}]}

print('titanic.csv.dvc (lo que va a git, ~200 bytes):')
for out in dvc_pointer['outs']:
    print(f"  md5:  {out['md5']}")
    print(f"  size: {out['size']}")
    print(f"  path: {out['path']}")

assert len(md5) == 32, 'un md5 hex tiene 32 chars'
assert dvc_pointer['outs'][0]['size'] == len(blob)
print('\nOK — el .dvc es un puntero por contenido; el blob (', len(blob), 'bytes) vive en el cache/remote.')

### Ejercicio 2 — Remote local y layout del blob

`dvc remote add -d local /tmp/dvc-remote-demo` + `dvc push` copia el blob al remote bajo `files/md5/<2-char-prefix>/<resto>`. Ese *content-addressed layout* (los 2 primeros chars del hash como carpeta) es idéntico al `.git/objects`: evita millones de archivos en un solo directorio. Lo reproducimos para ver dónde caería exactamente el blob del ejercicio 1.

In [ ]:
import shutil
from pathlib import Path

CLI = """
dvc remote add -d local /tmp/dvc-remote-demo
dvc push
ls /tmp/dvc-remote-demo/files/md5/        # <2-char>/<resto-del-hash>
"""

# concepto ejecutable: simulamos el push al remote local con el md5 del ej.1
remote = Path(tempfile.mkdtemp()) / 'dvc-remote-demo'
prefix, rest = md5[:2], md5[2:]
blob_dst = remote / 'files' / 'md5' / prefix / rest
blob_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(csv_path, blob_dst)

print('remote:', remote)
print('blob en:', blob_dst.relative_to(remote))
assert blob_dst.exists(), 'el blob debe existir tras el push simulado'
assert blob_dst.name == md5[2:] and blob_dst.parent.name == md5[:2]
print('OK — layout content-addressed:', f'files/md5/{prefix}/{rest[:8]}...')

### Ejercicio 3 — Pipeline declarativo (`dvc.yaml`) con dos stages

Definimos `prepare` y `train` en `dvc.yaml`. DVC re-ejecuta un stage solo si el hash de sus `deps` cambió (como `make`, pero por contenido). Escribimos el YAML **correcto** (validado con `pyyaml`) y además ejecutamos *la lógica real de cada stage* con scikit-learn, para ver los artefactos (`clean.csv`, `model.pkl`, `metrics.json`) que DVC orquestaría.

In [ ]:
import yaml, json, pickle
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

dvc_yaml = {
    'stages': {
        'prepare': {
            'cmd': 'python src/prepare.py',
            'deps': ['data/raw/titanic.csv', 'src/prepare.py'],
            'outs': ['data/processed/clean.csv'],
        },
        'train': {
            'cmd': 'python src/train.py',
            'deps': ['data/processed/clean.csv', 'src/train.py'],
            'params': ['test_size', 'random_state', 'model.C'],
            'outs': ['model.pkl'],
            'metrics': [{'metrics.json': {'cache': False}}],
        },
    }
}
# validamos que es YAML bien formado (dvc lo parsea igual)
assert yaml.safe_load(yaml.safe_dump(dvc_yaml)) == dvc_yaml
print('dvc.yaml válido:\n')
print(yaml.safe_dump(dvc_yaml, sort_keys=False))

# --- ejecución REAL de la lógica de los stages ---
params = {'test_size': 0.2, 'random_state': 42, 'model': {'C': 1.0}}
X, y = make_classification(n_samples=800, n_features=6, random_state=params['random_state'])
# stage prepare: (aquí solo simulamos "dropna" — el dataset ya está limpio)
clean_n = len(X)
# stage train:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=params['test_size'],
                                       random_state=params['random_state'])
model = LogisticRegression(C=params['model']['C'], max_iter=1000).fit(Xtr, ytr)
metrics = {'accuracy': round(accuracy_score(yte, model.predict(Xte)), 4),
           'f1': round(f1_score(yte, model.predict(Xte)), 4)}
print('metrics.json:', json.dumps(metrics))
assert 0.0 <= metrics['accuracy'] <= 1.0 and 'f1' in metrics
print('OK — prepare→train produjeron clean.csv (', clean_n, 'filas), model.pkl y metrics.json')

### Ejercicio 4 — Reproducción: por qué cambiar `test_size` re-ejecuta `train`

DVC decide re-ejecutar un stage comparando el hash de sus `deps`/`params` contra `dvc.lock`. `prepare` no depende de `params.yaml`, así que su hash no cambia; `train` sí lista `test_size` en `params`, así que al pasar `0.2 → 0.3` su hash cambia y el stage se re-ejecuta (arrastrando su output). Simulamos exactamente esa lógica de invalidación por hash.

In [ ]:
def stage_hash(deps: dict) -> str:
    """Hash determinístico de las deps de un stage — lo que DVC guarda en dvc.lock."""
    return hashlib.md5(json.dumps(deps, sort_keys=True).encode()).hexdigest()

# hashes con params originales (test_size=0.2)
lock_prepare_v1 = stage_hash({'raw': 'titanic-md5-fijo'})
lock_train_v1   = stage_hash({'clean': 'clean-md5', 'test_size': 0.2, 'C': 1.0})

# cambiamos SOLO test_size (0.2 -> 0.3)
lock_prepare_v2 = stage_hash({'raw': 'titanic-md5-fijo'})              # prepare no ve params
lock_train_v2   = stage_hash({'clean': 'clean-md5', 'test_size': 0.3, 'C': 1.0})

prepare_rerun = lock_prepare_v1 != lock_prepare_v2
train_rerun   = lock_train_v1 != lock_train_v2
print(f'prepare re-ejecuta? {prepare_rerun}   (esperado: False)')
print(f'train   re-ejecuta? {train_rerun}   (esperado: True)')
assert prepare_rerun is False, 'prepare no depende de params -> su hash no cambia'
assert train_rerun is True, 'train depende de test_size -> su hash cambia -> re-run'
print("\n`dvc repro --dry` mostraría: Running stage 'train' (prepare está 'up to date').")

### Ejercicio 5 — Experimentos sin branching (`dvc exp run`)

`dvc exp run -S 'model.C=0.1'` corre el pipeline con un override de parámetro y guarda el resultado como un experimento (commit interno, no ensucia `git log`). `dvc exp show` los tabula. Replicamos el sweep de `C` en Python: corremos 3 valores, tabulamos como haría `dvc exp show`, y elegimos el mejor (`dvc exp apply`).

In [ ]:
CLI = """
dvc exp run -S 'model.C=0.1'
dvc exp run -S 'model.C=1'
dvc exp run -S 'model.C=10'
dvc exp show --no-pager
dvc exp apply <hash-del-mejor>
"""

experiments = []
for C in [0.1, 1.0, 10.0]:
    m = LogisticRegression(C=C, max_iter=1000).fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    experiments.append({'exp': f'C={C}', 'model.C': C, 'accuracy': round(acc, 4)})

print(f'{"Experiment":12} {"model.C":>8} {"accuracy":>10}   <- dvc exp show')
for e in experiments:
    print(f'{e["exp"]:12} {e["model.C"]:>8} {e["accuracy"]:>10}')

best = max(experiments, key=lambda e: e['accuracy'])
print(f'\nmejor: {best["exp"]}  (dvc exp apply lo promovería al workspace)')
assert best['accuracy'] == max(e['accuracy'] for e in experiments)
print('OK — sweep de C sin crear una sola rama git.')